In [1]:
# ============================ INSTALL ============================
!pip install -q timm==1.0.3

# ============================ IMPORTS ============================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import timm
from torch.cuda.amp import autocast, GradScaler
import pandas as pd
from PIL import Image
from tqdm import tqdm
import gc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 69.4 MB/s eta 0:00:00


In [2]:
# ============================ CONFIG ============================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 38
BATCH_SIZE = 80
EPOCHS_STAGE1 = 3      # Head training
EPOCHS_STAGE2 = 8      # Full fine-tuning
IMG_SIZE = 256
NUM_WORKERS = 4
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

TRAIN_ROOT = "/kaggle/input/competitions/dlp-image-classification-apr-2026-2/train"
TEST_ROOT = "/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test"

In [3]:
torch.cuda.empty_cache()
gc.collect()

print(f"Using {torch.cuda.device_count()} GPU(s) — {DEVICE}")

Using 2 GPU(s) — cuda


In [4]:
# ============================ TRANSFORMS ============================
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=7),      # ← big accuracy boost, almost free
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [5]:
# ============================ DATASETS & LOADERS ============================
train_dataset = datasets.ImageFolder(root=TRAIN_ROOT, transform=train_transform)
print(f"Training images: {len(train_dataset)}")
print("Class order (should match 0-37):", train_dataset.classes[:5], "...", train_dataset.classes[-5:])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)

class TestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        
        # Get all image files (handles both .jpg and .JPG)
        all_files = [f for f in os.listdir(img_dir) 
                     if f.lower().endswith(('.jpg', '.jpeg'))]
        
        # Sort NUMERICALLY by extracting the number safely
        def get_image_number(filename):
            # Remove 'image_' and extension (handles .jpg and .JPG)
            clean = filename.replace('image_', '').strip()
            clean = clean.replace('.jpg', '').replace('.JPG', '').replace('.jpeg', '').replace('.JPEG', '')
            return int(clean)
        
        self.image_files = sorted(all_files, key=get_image_number)
        
        print(f" Total test images found: {len(self.image_files)}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        image_id = img_name.replace('.jpg', '').replace('.JPG', '')   # Clean ID
        if self.transform:
            image = self.transform(image)
        return image, image_id

test_dataset = TestDataset(TEST_ROOT, transform=test_transform)
test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=NUM_WORKERS, 
    pin_memory=True
)

print(f"Testing images: {len(test_dataset)}")

# ============================ MODEL ============================
MODEL_NAME = 'tf_efficientnet_b3'   # Best speed/accuracy balance for this task
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

print(f"Model loaded: {MODEL_NAME} | Parameters: {sum(p.numel() for p in model.parameters()):,}")

Training images: 43429
Class order (should match 0-37): ['Apple_Apple_scab', 'Apple_Black_rot', 'Apple_Cedar_apple_rust', 'Apple_healthy', 'Blueberry_healthy'] ... ['Tomato_Spider_mites_Two-spotted_spider_mite', 'Tomato_Target_Spot', 'Tomato_Tomato_Yellow_Leaf_Curl_Virus', 'Tomato_Tomato_mosaic_virus', 'Tomato_healthy']
 Total test images found: 10876
Testing images: 10876


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Model loaded: tf_efficientnet_b3 | Parameters: 10,754,638


In [6]:
torch.cuda.empty_cache()  
gc.collect()

145

In [7]:
# ============================ STAGE 1: Train Head Only ============================
print("\n=== STAGE 1: Training only the classifier head ===")

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the head (works for tf_efficientnet, convnext, etc.)
for name, param in model.named_parameters():
    if 'classifier' in name or 'head' in name or 'fc' in name:
        param.requires_grad = True

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = GradScaler()

for epoch in range(EPOCHS_STAGE1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Stage 1 Epoch {epoch+1}/{EPOCHS_STAGE1}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    print(f"Stage 1 Epoch {epoch+1}: Loss = {running_loss/len(train_loader):.4f} | Acc = {acc:.2f}%")

/tmp/ipykernel_22/3265954480.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



=== STAGE 1: Training only the classifier head ===


Stage 1 Epoch 1/3:   0%|          | 0/542 [00:00<?, ?it/s]/tmp/ipykernel_22/3265954480.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Stage 1 Epoch 1/3: 100%|██████████| 542/542 [02:32<00:00,  3.57it/s]


Stage 1 Epoch 1: Loss = 0.8834 | Acc = 86.58%


Stage 1 Epoch 2/3: 100%|██████████| 542/542 [02:01<00:00,  4.48it/s]


Stage 1 Epoch 2: Loss = 0.6559 | Acc = 93.55%


Stage 1 Epoch 3/3: 100%|██████████| 542/542 [01:58<00:00,  4.57it/s]

Stage 1 Epoch 3: Loss = 0.6093 | Acc = 95.03%


In [8]:
# ============================ STAGE 2: Full Fine-tuning ============================
print("\n=== STAGE 2: Fine-tuning entire model ===")

# Unfreeze everything
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

for epoch in range(EPOCHS_STAGE2):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Stage 2 Epoch {epoch+1}/{EPOCHS_STAGE2}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    scheduler.step()

    acc = 100. * correct / total
    print(f"Stage 2 Epoch {epoch+1}: Loss = {running_loss/len(train_loader):.4f} | Acc = {acc:.2f}%")


=== STAGE 2: Fine-tuning entire model ===


Stage 2 Epoch 1/8:   0%|          | 0/542 [00:00<?, ?it/s]/tmp/ipykernel_22/3722367354.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Stage 2 Epoch 1/8: 100%|██████████| 542/542 [05:17<00:00,  1.71it/s]


Stage 2 Epoch 1: Loss = 0.5248 | Acc = 97.53%


Stage 2 Epoch 2/8: 100%|██████████| 542/542 [04:35<00:00,  1.97it/s]


Stage 2 Epoch 2: Loss = 0.4773 | Acc = 98.75%


...

Stage 2 Epoch 6: Loss = 0.4275 | Acc = 99.65%


Stage 2 Epoch 7/8: 100%|██████████| 542/542 [04:35<00:00,  1.97it/s]


Stage 2 Epoch 7: Loss = 0.4246 | Acc = 99.61%


Stage 2 Epoch 8/8: 100%|██████████| 542/542 [04:35<00:00,  1.97it/s]

Stage 2 Epoch 8: Loss = 0.4220 | Acc = 99.73%


In [9]:
# ============================ SAVE MODEL WEIGHTS ============================
import os

# Define save path explicitly in /kaggle/working/
SAVE_DIR = "/kaggle/working/"
os.makedirs(SAVE_DIR, exist_ok=True)   # Just in case

WEIGHTS_PATH = os.path.join(SAVE_DIR, "plant_disease_weights.pth")

# Best & Cleanest way - Save only weights
torch.save(model.state_dict(), WEIGHTS_PATH)

print(" Model weights saved successfully!")
print(f"Path: {WEIGHTS_PATH}")
print(f"File size: {os.path.getsize(WEIGHTS_PATH) / (1024*1024):.1f} MB")

# Optional: Also save a small checkpoint with metadata (very useful)
checkpoint = {
    'model_name': MODEL_NAME,
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
}

checkpoint_path = os.path.join(SAVE_DIR, "plant_disease_checkpoint.pth")
torch.save(checkpoint, checkpoint_path)

print(f"Checkpoint also saved at: {checkpoint_path}")

 Model weights saved successfully!
Path: /kaggle/working/plant_disease_weights.pth
File size: 41.6 MB
Checkpoint also saved at: /kaggle/working/plant_disease_checkpoint.pth


In [10]:
torch.cuda.empty_cache()   
gc.collect()

18

In [11]:
# # ============================ LOAD WEIGHTS ============================
# model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
# model = model.to(DEVICE)

# # Load the saved weights
# weights = torch.load('plant_disease_weights.pth', map_location=DEVICE)
# model.load_state_dict(weights)

# model.eval()
# print(" Model weights loaded successfully!")

In [12]:
# ============================ INFERENCE ============================
print("\n=== Running inference on full test set ===")
model.eval()
predictions = []

with torch.no_grad():
    for images, image_ids in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE)
        with autocast():
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
        
        for img_id, pred in zip(image_ids, preds.cpu().numpy()):
            predictions.append({'Image_ID': img_id, 'Label': int(pred)})


=== Running inference on full test set ===


Inference:   0%|          | 0/136 [00:00<?, ?it/s]/tmp/ipykernel_22/1499641205.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Inference: 100%|██████████| 136/136 [00:43<00:00,  3.09it/s]


In [13]:
# Create submission
submission_df = pd.DataFrame(predictions)

# Important: Sort by Image_ID to match the expected order
submission_df = submission_df.sort_values(by='Image_ID').reset_index(drop=True)

submission_df.to_csv('submission.csv', index=False)

print("\n Final Submission saved!")
print(f"Total predictions: {len(submission_df)}")   
print(submission_df.head(10))
print(submission_df.tail(5))


 Final Submission saved!
Total predictions: 10876
      Image_ID  Label
0  image_00001      0
1  image_00002      0
2  image_00003      0
3  image_00004      0
4  image_00005      0
5  image_00006      0
6  image_00007      0
7  image_00008      0
8  image_00009      0
9  image_00010      0
          Image_ID  Label
10871  image_10872     37
10872  image_10873     37
10873  image_10874     37
10874  image_10875     37
10875  image_10876     37
